# FedPLDSE 联邦学习复现（内存优化版）
论文：《FedPLDSE: 面向异构智慧城市设备的联邦学习子模型提取》

**按顺序运行每个 Cell 即可。**

## Cell 1 — 安装依赖

In [1]:
!pip install matplotlib tqdm -q
!pip install "numpy<2"
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


## Cell 2 — ⚙️ 参数配置（只需改这里）

| 参数 | 说明 | 推荐值 |
|------|------|--------|
| `DATASET` | 数据集 | `'cifar10'` / `'cifar100'` / `'fashion_mnist'` |
| `ALPHA` | 数据异构程度 | `0.5`=低异构，`0.1`=高异构 |
| `DEVICE_CONFIG` | 设备算力配置 | `'a-b-c-d-e'`(完整) / `'e'`(全低算力) |
| `IMPORTANT_RATIO` | 每层重要参数比例 | 论文最优 `0.05` |
| `ROUNDS` | 全局通信轮数 | cifar10→`500`，cifar100→`1200` |
| `NUM_CLIENTS` | 客户端总数 | `20`（内存紧张可改 `10`） |
| `CLIENTS_PER_ROUND` | 每轮参与客户端数 | `10`（内存紧张可改 `5`） |
| `LOCAL_EPOCHS` | 每轮本地训练轮数 | `5`（内存紧张可改 `2`） |
| `LOCAL_BATCH_SIZE` | 本地批大小 | `32`（内存紧张可改 `16`） |
| `LR` | 学习率 | `0.01` |

In [2]:
# 本地测试专用：极致精简，1分钟内跑完，仅验证代码无语法/逻辑错误
DATASET           = 'cifar10'
ALPHA             = 0.5
DEVICE_CONFIG     = 'e'          # 全低算力，减少计算量
IMPORTANT_RATIO   = 0.05

# 核心精简参数（本地测试唯一需要改的部分）
ROUNDS            = 1            # 只跑1轮，最快验证
NUM_CLIENTS       = 1            # 仅1个客户端，无多客户端通信开销
CLIENTS_PER_ROUND = 1            # 每轮仅1个客户端参与
LOCAL_EPOCHS      = 1            # 本地仅训练1轮
LOCAL_BATCH_SIZE  = 4            # 批大小压到最小，CPU/GPU都无压力
LR                = 0.01
MOMENTUM          = 0.9
WEIGHT_DECAY      = 1e-4

LOG_EVERY         = 1            # 每轮打印，确认代码在运行
DATA_ROOT         = './data'     # 数据集本地存放路径
SEED              = 42

## Cell 3 — 导入库 & 选择设备

In [3]:
import gc
import random
import numpy as np
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'使用设备: {DEVICE}')

使用设备: cuda


## Cell 4 — 模型 + 数据 + 工具函数

In [4]:
# ──────────────────────────────────────
# ResNet-18（GroupNorm 版）
# ──────────────────────────────────────
class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1    = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1      = nn.GroupNorm(min(2, planes), planes)
        self.conv2    = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.bn2      = nn.GroupNorm(min(2, planes), planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.GroupNorm(min(2, planes), planes),
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_planes = 64
        self.conv1  = nn.Conv2d(3, 64, 3, padding=1, bias=False)
        self.bn1    = nn.GroupNorm(2, 64)
        self.layer1 = self._make(64,  2, 1)
        self.layer2 = self._make(128, 2, 2)
        self.layer3 = self._make(256, 2, 2)
        self.layer4 = self._make(512, 2, 2)
        self.fc     = nn.Linear(512, num_classes)
    def _make(self, planes, n, stride):
        layers = [BasicBlock(self.in_planes, planes, stride)]
        self.in_planes = planes
        for _ in range(n - 1):
            layers.append(BasicBlock(planes, planes))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.fc(F.adaptive_avg_pool2d(x, 1).view(x.size(0), -1))

def build_model(dataset):
    nc = {'cifar10': 10, 'cifar100': 100, 'fashion_mnist': 10}[dataset]
    return ResNet18(num_classes=nc)

# ──────────────────────────────────────
# 数据加载
# ──────────────────────────────────────
def load_dataset(dataset, root='./data'):
    if dataset in ['cifar10', 'cifar100']:
        mu, sg = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
        tr = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                  transforms.RandomHorizontalFlip(),
                                  transforms.ToTensor(),
                                  transforms.Normalize(mu, sg)])
        te = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mu, sg)])
        cls = datasets.CIFAR10 if dataset == 'cifar10' else datasets.CIFAR100
    else:
        tr = transforms.Compose([transforms.Resize(32), transforms.Grayscale(3),
                                  transforms.RandomHorizontalFlip(),
                                  transforms.ToTensor(),
                                  transforms.Normalize((0.286,), (0.353,))])
        te = transforms.Compose([transforms.Resize(32), transforms.Grayscale(3),
                                  transforms.ToTensor(),
                                  transforms.Normalize((0.286,), (0.353,))])
        cls = datasets.FashionMNIST
    return (cls(root, train=True,  download=True, transform=tr),
            cls(root, train=False, download=True, transform=te))

def dirichlet_partition(dataset, n_clients, alpha, seed=42):
    np.random.seed(seed)
    targets = np.array(dataset.targets)
    n_cls   = len(np.unique(targets))
    cls_idx = [np.where(targets == c)[0] for c in range(n_cls)]
    client_idx = [[] for _ in range(n_clients)]
    for c in range(n_cls):
        np.random.shuffle(cls_idx[c])
        props = np.random.dirichlet(np.repeat(alpha, n_clients))
        props = (props * len(cls_idx[c])).astype(int)
        props[-1] = len(cls_idx[c]) - props[:-1].sum()
        props = np.maximum(props, 0)
        s = 0
        for i, cnt in enumerate(props):
            client_idx[i].extend(cls_idx[c][s:s+cnt].tolist())
            s += cnt
    return client_idx

CAPACITY_MAP = {'a':1.0,'b':0.5,'c':0.25,'d':0.125,'e':0.0625}
CONFIG_MAP   = {'a-b-c-d-e':list('abcde'),'b-c-d-e':list('bcde'),
                'c-d-e':list('cde'),'d-e':list('de'),'e':['e']}

def assign_capacities(n, config, seed=42):
    np.random.seed(seed)
    lvls = CONFIG_MAP[config]
    caps = [CAPACITY_MAP[lvls[i % len(lvls)]] for i in range(n)]
    np.random.shuffle(caps)
    return caps

print('模型 & 数据工具定义完成 ✓')

模型 & 数据工具定义完成 ✓


## Cell 5 — 加载数据

In [5]:
train_set, test_set = load_dataset(DATASET, DATA_ROOT)

# Windows 上 num_workers 必须设 0，否则多进程会崩溃
NW = 0 if __import__('platform').system() == 'Windows' else 2

# 服务端验证集（训练集+测试集各取10%）
rng   = np.random.RandomState(SEED)
t_idx = rng.choice(len(train_set), int(0.1*len(train_set)), replace=False).tolist()
e_idx = rng.choice(len(test_set),  int(0.1*len(test_set)),  replace=False).tolist()
val_loader  = DataLoader(ConcatDataset([Subset(train_set, t_idx), Subset(test_set, e_idx)]),
                         batch_size=4, shuffle=False, num_workers=NW)
test_loader = DataLoader(test_set, batch_size=4, shuffle=False, num_workers=NW)

# 客户端数据划分
client_indices = dirichlet_partition(train_set, NUM_CLIENTS, ALPHA, SEED)
client_loaders = [
    DataLoader(Subset(train_set, idx), batch_size=LOCAL_BATCH_SIZE,
               shuffle=True, num_workers=NW,
               drop_last=len(idx) > LOCAL_BATCH_SIZE)
    for idx in client_indices
]

capacities = assign_capacities(NUM_CLIENTS, DEVICE_CONFIG, SEED)

print(f'训练集: {len(train_set)}  测试集: {len(test_set)}  验证集: {len(val_loader.dataset)}')
print(f'num_workers={NW}  客户端算力: {sorted(set(capacities), reverse=True)}')
print('数据加载完成 ✓')

Files already downloaded and verified
Files already downloaded and verified
训练集: 50000  测试集: 10000  验证集: 6000
num_workers=0  客户端算力: [0.0625]
数据加载完成 ✓


## Cell 6 — 核心算法（服务端 + 客户端）

In [6]:
# ══════════════════════════════════════════════════════
#  服务端
# ══════════════════════════════════════════════════════
class PLDSEServer:
    def __init__(self, global_model, device, val_loader, important_ratio=0.05):
        self.model    = global_model.to(device)
        self.device   = device
        self.val_loader = val_loader
        self.imp_ratio  = important_ratio
        self.loss_fn    = nn.CrossEntropyLoss()
        self.round      = 0
        # 记录每个参数层的参数数量
        self.layer_sizes = {n: p.numel()
                            for n, p in self.model.named_parameters()
                            if p.requires_grad}

    # ── 1. 参数重要性评分（公式 13、17）──────────────────────
# ── 1. 参数重要性评分（公式 13、17）──────────────────────
    def compute_importance(self):
        """单次前向+反向，S_j = |∇L_j|²，层内归一化。"""
        self.model.eval()
        self.model.zero_grad()

        # 获取验证集总样本数
        n = len(self.val_loader.dataset)

        # 逐批次进行前向和反向传播，避免计算图在显存中堆积
        for x, y in self.val_loader:
            x, y = x.to(self.device), y.to(self.device)

            # 计算当前批次的 loss，并按样本比例缩放
            loss = self.loss_fn(self.model(x), y) * (x.size(0) / n)

            # ❗立刻反向传播，累加梯度，并瞬间释放该批次的显存计算图
            loss.backward()

        scores = {}
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            if param.grad is None:
                scores[name] = torch.zeros(param.numel(), device=self.device)
            else:
                s = param.grad.detach().pow(2).view(-1)
                s_sum = s.sum()
                scores[name] = (s / s_sum * 100.0) if s_sum > 0 else s

        self.model.zero_grad()
        return scores
    # ── 2. 子模型索引（公式 18、19、20）─────────────────────
    def get_masks(self, capacity, scores):
        """返回每层的布尔掩码：重要参数 ∪ 滚动参数。"""
        masks, t = {}, self.round
        for name, K in self.layer_sizes.items():
            Ki  = max(1, int(capacity * K))
            Ko  = min(max(1, int(np.ceil(self.imp_ratio * K))), Ki)
            Kr  = Ki - Ko

            s = scores.get(name, torch.zeros(K, device=self.device))
            _, top_idx = torch.topk(s, Ko)
            important  = set(top_idx.cpu().numpy().tolist())

            rolling, cand = set(), t % K
            while len(rolling) < Kr:
                if cand not in important:
                    rolling.add(cand)
                cand = (cand + 1) % K

            mask = torch.zeros(K, dtype=torch.bool, device=self.device)
            for idx in important | rolling:
                mask[idx] = True
            masks[name] = mask
        return masks

    # ── 3. 提取子模型权重（公式 21）─────────────────────────
    def get_submodel_weights(self, masks):
        """返回全局模型当前权重的 dict（非选中位置置零）。"""
        sub = {}
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if not param.requires_grad:
                    continue
                w = param.data.clone()
                w.view(-1)[~masks[name]] = 0.0
                sub[name] = w
        return sub

    # ── 4. 稀疏聚合（公式 22）───────────────────────────────
    def aggregate(self, updates):
        """对每个参数，只对更新过它的客户端取平均。"""
        param_sum   = {n: torch.zeros_like(p.data)
                       for n, p in self.model.named_parameters() if p.requires_grad}
        param_count = {n: torch.zeros(k, device=self.device)
                       for n, k in self.layer_sizes.items()}
        for delta, masks in updates:
            for name in param_sum:
                if name not in delta:
                    continue
                param_sum[name]   += delta[name]
                param_count[name] += masks[name].float()
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if not param.requires_grad:
                    continue
                flat  = param.data.view(-1)
                count = param_count[name]
                upd   = count > 0
                flat[upd] += param_sum[name].view(-1)[upd] / count[upd]
                param.data.copy_(flat.view(param.shape))
        # 清理聚合用的临时变量
        del param_sum, param_count
        gc.collect()
        self.round += 1

    # ── 评估 ────────────────────────────────────────────────
    def evaluate(self, loader):
        self.model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                correct += self.model(x).argmax(1).eq(y).sum().item()
                total   += y.size(0)
        return 100.0 * correct / total


# ══════════════════════════════════════════════════════
#  客户端（无 deepcopy，直接用全局模型权重覆写本地模型）
# ══════════════════════════════════════════════════════
class FedPLDSEClient:
    """每个客户端持有一个固定的本地模型实例，避免反复 deepcopy。"""

    def __init__(self, capacity, local_loader, device,
                 lr=0.01, momentum=0.9, weight_decay=1e-4, local_epochs=5):
        self.capacity     = capacity
        self.local_loader = local_loader
        self.device       = device
        self.local_epochs = local_epochs
        self.loss_fn      = nn.CrossEntropyLoss()
        # 每个客户端只创建一次本地模型
        self.local_model  = None
        self.lr = lr; self.momentum = momentum; self.wd = weight_decay

    def setup(self, global_model):
        """首次初始化时复制全局模型架构，之后复用。"""
        if self.local_model is None:
            import copy
            self.local_model = copy.deepcopy(global_model).to(self.device)
            self.optimizer   = optim.SGD(self.local_model.parameters(),
                                         lr=self.lr, momentum=self.momentum,
                                         weight_decay=self.wd)

    def train(self, global_weights, masks):
        """
        加载子模型权重 → 本地 SGD → 返回参数增量 delta。
        不做 deepcopy，直接用 load_state_dict 覆写，节省内存。
        """
        model = self.local_model

        # 用全局权重初始化本地模型（只覆写可训练参数）
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name in global_weights:
                    param.data.copy_(global_weights[name].to(self.device))

        # 记录初始权重（仅记录选中参数，节省内存）
        init = {}
        with torch.no_grad():
            for name, param in model.named_parameters():
                if param.requires_grad and name in masks:
                    # 只存选中参数的值，用 CPU 存储节省显存
                    m = masks[name]
                    init[name] = param.data.view(-1)[m].cpu().clone()

        # 本地训练
        model.train()
        for _ in range(self.local_epochs):
            for x, y in self.local_loader:
                x, y = x.to(self.device), y.to(self.device)
                self.optimizer.zero_grad()
                self.loss_fn(model(x), y).backward()
                # 非选中参数梯度置零
                with torch.no_grad():
                    for name, param in model.named_parameters():
                        if param.requires_grad and param.grad is not None and name in masks:
                            param.grad.view(-1)[~masks[name]] = 0.0
                self.optimizer.step()
                # 非选中参数恢复初始值
                with torch.no_grad():
                    for name, param in model.named_parameters():
                        if param.requires_grad and name in masks:
                            m = masks[name]
                            param.data.view(-1)[~m] = global_weights[name].to(self.device).view(-1)[~m]

        # 计算增量（只返回选中参数的增量，CPU 存储）
        delta = {}
        with torch.no_grad():
            for name, param in model.named_parameters():
                if not param.requires_grad or name not in masks:
                    continue
                m = masks[name]
                d = torch.zeros(param.numel(), device='cpu')
                d[m] = param.data.view(-1)[m].cpu() - init[name]
                delta[name] = d.view(param.shape)

        del init
        return delta, masks

print('服务端 & 客户端定义完成 ✓')

服务端 & 客户端定义完成 ✓


## Cell 7 — 初始化

In [7]:
global_model = build_model(DATASET)

server = PLDSEServer(
    global_model    = global_model,
    device          = DEVICE,
    val_loader      = val_loader,
    important_ratio = IMPORTANT_RATIO,
)

clients = [
    FedPLDSEClient(
        capacity     = capacities[i],
        local_loader = client_loaders[i],
        device       = DEVICE,
        lr           = LR,
        momentum     = MOMENTUM,
        weight_decay = WEIGHT_DECAY,
        local_epochs = LOCAL_EPOCHS,
    )
    for i in range(NUM_CLIENTS)
]

# 提前初始化每个客户端的本地模型（只做一次 deepcopy）
for c in clients:
    c.setup(server.model)

n_params = sum(p.numel() for p in server.model.parameters())
print(f'全局模型参数量: {n_params:,}')
print(f'客户端数量: {NUM_CLIENTS}')
print('初始化完成 ✓')

全局模型参数量: 11,173,962
客户端数量: 1
初始化完成 ✓


D:\Gracduation project\.venv\Lib\site-packages\torch\nn\modules\module.py:1160: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


## Cell 8 — 开始训练 🚀

In [8]:
## Cell 8 — 开始训练 🚀
import time
import sys
# 替换notebook版tqdm为通用文本版，兼容所有环境
from tqdm import tqdm

set_seed(SEED)
best_acc = 0.0
history = []

# ── 预热1轮（打印详细步骤日志）─────────────────────────────────────
print('=' * 60)
print('开始预热（仅1轮）- 详细日志：')
print('=' * 60)
warmup_times = []
warmup_rounds = 1  # 固定预热1轮

for r in range(warmup_rounds):
    round_start = time.time()
    print(f'\n【预热轮次 {r + 1}/{warmup_rounds}】')

    # 步骤1：计算参数重要性（打印耗时）
    print('  步骤1/5：计算参数重要性...', end='')
    scores_start = time.time()
    scores = server.compute_importance()
    scores_time = time.time() - scores_start
    print(f' 完成！耗时 {scores_time:.2f} 秒')

    # 步骤2：选择客户端（打印选中的客户端ID）
    print('  步骤2/5：选择参与客户端...', end='')
    selected = random.sample(range(NUM_CLIENTS), min(CLIENTS_PER_ROUND, NUM_CLIENTS))
    print(f' 完成！选中客户端ID：{selected}')

    # 步骤3：客户端训练（文本进度条，无ipywidgets依赖）
    print(f'  步骤3/5：客户端训练（共{len(selected)}个客户端）...')
    updates = []
    # 通用文本版进度条，兼容本地/服务器
    client_pbar = tqdm(selected, desc='    客户端进度', leave=False, file=sys.stdout)
    for cid in client_pbar:
        client_start = time.time()
        c = clients[cid]
        # 获取该客户端的掩码和子模型权重
        masks = server.get_masks(c.capacity, scores)
        sub_w = server.get_submodel_weights(masks)
        # 客户端本地训练
        delta, m = c.train(sub_w, masks)
        updates.append((delta, m))
        del sub_w  # 释放内存
        # 打印该客户端耗时
        client_time = time.time() - client_start
        client_pbar.set_postfix(客户端ID=cid, 耗时=f'{client_time:.2f}s')
    client_pbar.close()
    print(f'  步骤3/5：客户端训练 完成！')

    # 步骤4：服务端聚合（打印耗时）
    print('  步骤4/5：服务端参数聚合...', end='')
    agg_start = time.time()
    del scores  # 释放内存
    server.aggregate(updates)
    del updates  # 释放内存
    agg_time = time.time() - agg_start
    print(f' 完成！耗时 {agg_time:.2f} 秒')

    # 步骤5：清理内存（打印显存状态）
    print('  步骤5/5：清理内存/显存...', end='')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        # 打印GPU显存使用（无需psutil，用torch原生接口）
        gpu_mem = torch.cuda.memory_allocated() / 1024 / 1024
        print(f' 完成！GPU已用显存：{gpu_mem:.1f} MB')
    else:
        print(' 完成！（使用CPU，无显存）')

    # 记录本轮总耗时
    round_total_time = time.time() - round_start
    warmup_times.append(round_total_time)
    print(f'\n【预热轮次 {r + 1} 完成】总耗时：{round_total_time:.2f} 秒')

# 预热完成后计算预估时间
print('\n' + '=' * 60)
print('预热完成！')
print('=' * 60)
sec_per_round = sum(warmup_times) / len(warmup_times)
remaining = ROUNDS - warmup_rounds
eta_min = sec_per_round * remaining / 60
eta_hr = eta_min / 60
print(f'✅ 预热单轮耗时：{sec_per_round:.1f} 秒')
if remaining > 0:
    if eta_hr >= 1:
        print(f'⏱️  预计剩余训练时间：{eta_hr:.1f} 小时（共 {ROUNDS} 轮）')
    else:
        print(f'⏱️  预计剩余训练时间：{eta_min:.0f} 分钟（共 {ROUNDS} 轮）')
else:
    print(f'⏱️  无剩余轮次（总轮数={ROUNDS}）')

# ── 正式训练循环（逻辑不变）──────────────────────────────────────────
print('\n' + '=' * 60)
print('开始正式训练：')
print('=' * 60)
t_start = time.time()
pbar = tqdm(range(warmup_rounds, ROUNDS), desc='训练中', unit='轮',
            initial=warmup_rounds, total=ROUNDS, file=sys.stdout)

for r in pbar:
    t0 = time.time()

    scores = server.compute_importance()
    selected = random.sample(range(NUM_CLIENTS), min(CLIENTS_PER_ROUND, NUM_CLIENTS))
    updates = []
    for cid in selected:
        c = clients[cid]
        masks = server.get_masks(c.capacity, scores)
        sub_w = server.get_submodel_weights(masks)
        delta, m = c.train(sub_w, masks)
        updates.append((delta, m))
        del sub_w
    del scores
    server.aggregate(updates)
    del updates;
    gc.collect()

    round_sec = time.time() - t0

    if (r + 1) % LOG_EVERY == 0:
        acc = server.evaluate(test_loader)
        best_acc = max(best_acc, acc)
        history.append({'round': r + 1, 'acc': acc})
        elapsed = (time.time() - t_start) / 60
        eta_left = round_sec * (ROUNDS - r - 1) / 60
        pbar.set_postfix(
            精度=f'{acc:.2f}%',
            最佳=f'{best_acc:.2f}%',
            已用=f'{elapsed:.0f}min',
            剩余=f'{eta_left:.0f}min'
        )
        print(f'  轮次[{r + 1:4d}/{ROUNDS}]  精度:{acc:.2f}%  最佳:{best_acc:.2f}%'
              f'  已用:{elapsed:.1f}min  剩余:{eta_left:.0f}min')

pbar.close()

# 最终评估
final_acc = server.evaluate(test_loader)
best_acc = max(best_acc, final_acc)
total_min = (time.time() - t_start) / 60
print('=' * 55)
print(f'  训练完成！最终精度: {final_acc:.2f}%  最佳精度: {best_acc:.2f}%')
print(f'  总用时: {total_min:.1f} 分钟')
print('=' * 55)

开始预热（仅1轮）- 详细日志：

【预热轮次 1/1】
  步骤1/5：计算参数重要性... 完成！耗时 22.77 秒
  步骤2/5：选择参与客户端... 完成！选中客户端ID：[0]
  步骤3/5：客户端训练（共1个客户端）...
  步骤3/5：客户端训练 完成！                                                                       
  步骤4/5：服务端参数聚合...

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

## Cell 9 — 绘制训练曲线

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'SimHei'   # Windows 中文字体
matplotlib.rcParams['axes.unicode_minus'] = False

rounds_log = [h['round'] for h in history]
accs_log   = [h['acc']   for h in history]

plt.figure(figsize=(9, 4))
plt.plot(rounds_log, accs_log, color='steelblue', linewidth=1.5, label='全局精度')
plt.axhline(best_acc, color='tomato', linestyle='--', linewidth=1.2,
            label=f'最佳精度 {best_acc:.2f}%')
plt.xlabel('通信轮次')
plt.ylabel('测试精度 (%)')
plt.title(f'FedPLDSE — {DATASET} | α={ALPHA} | {DEVICE_CONFIG}')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_curve.png', dpi=150)
plt.show()
print('曲线已保存为 training_curve.png')